# Phase#2 — Multi-Brand Classifier — Statistics

In [1]:
import plotly.io as pio

import sys
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from IPython.display import display

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Data" / "original").exists() and (p / "Scripts").exists():
            return p
        if p.name.startswith("Phase#2"):
            return p
    raise RuntimeError("Cannot locate Phase#2 root. Start Jupyter from Main/ or Main/Phase#2/")

_PHASE2         = _find_root()
_LINEARSVC_DIR  = _PHASE2 / "Models" / "linearcvs"
_LABELED_SVC    = _PHASE2 / "Data" / "processed" / "1M_parts_numbers_labeled.csv"
_LABELED_TOPBFM = _PHASE2 / "Data" / "processed" / "1M_parts_numbers_labeled_topbfm.csv"
_CLUSTER_JSON   = _PHASE2 / "Reports" / "cluster_distribution.json"
_REPORTS_DIR    = _PHASE2 / "Reports"
print("Phase#2 root:", _PHASE2)

Phase#2 root: /home/cmdr-nikel/Documents/FsML_project-1/Main/Phase#2


## LinearSVC — Model Parameters

In [2]:
svc_path = sorted(_LINEARSVC_DIR.glob("*.pkl"))[-1]
bundle   = joblib.load(svc_path)
model    = bundle["model"]
inner    = model.estimator if hasattr(model, "estimator") else model

print(f"File     : {svc_path.name}")
print(f"Type     : CalibratedClassifierCV(LinearSVC)  |  C={inner.C}")
print(f"Classes  : {list(model.classes_)}")
print(f"Features : {len(bundle['feature_order'])}")
display(pd.DataFrame({"#": range(len(bundle['feature_order'])), "feature": bundle["feature_order"]}))

File     : linearsvc_atom_1184k_4cls.pkl
Type     : CalibratedClassifierCV(LinearSVC)  |  C=0.01
Classes  : ['bmw', 'mercedes', 'unknown_article', 'vag']
Features : 52


,#,feature
0,0,article_len
1,1,num_letters
2,2,num_digits
3,3,first_char_is_letter
4,4,prefix_letters
5,5,suffix_letters
6,6,suffix_digits
7,7,all_digits
8,8,digit_ratio
9,9,has_only_alnum


## LinearSVC — 1M File Results

In [3]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "nbformat"], check=True)


svc_df = pd.read_csv(_LABELED_SVC, dtype=str)
total  = len(svc_df)

counts = svc_df["label"].value_counts().reset_index()
counts.columns = ["label", "count"]
counts["pct"] = (counts["count"] / total * 100).round(2)

fig = px.bar(
    counts, x="label", y="count", text="pct",
    title=f"LinearSVC — label distribution ({total:,} articles)",
    color="label",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()

prob_cols = [c for c in svc_df.columns if c.endswith("_prob")]
if prob_cols:
    conf_rows = []
    for lbl in svc_df["label"].unique():
        sub = svc_df[svc_df["label"] == lbl][prob_cols].apply(pd.to_numeric, errors="coerce")
        mp  = sub.max(axis=1).dropna()
        if len(mp) == 0: continue
        conf_rows.append({"label": lbl, "mean": mp.mean(), "p10": mp.quantile(0.1), "p90": mp.quantile(0.9)})
    cdf = pd.DataFrame(conf_rows)
    cdf["err_high"] = cdf["p90"] - cdf["mean"]
    cdf["err_low"]  = cdf["mean"] - cdf["p10"]
    fig2 = px.bar(
        cdf, x="label", y="mean",
        error_y="err_high", error_y_minus="err_low",
        title="LinearSVC — mean confidence per label  [error bars: p10–p90]",
        color="label",
    )
    fig2.show()

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [nbformat]6/7 [nbformat]


## TopBFM — Training Report

In [4]:
reports = sorted(_REPORTS_DIR.glob("topbfm_*.txt"))
latest  = reports[-1]
print(f"Latest training report: {latest.name}  ({len(reports)} total)\n")
print(latest.read_text(encoding="utf-8"))

Latest training report: topbfm_20260515_171725.txt  (8 total)

n_clusters: 550
purity_threshold: 0.92
report:
                 precision    recall  f1-score   support

            bmw       1.00      0.95      0.97    120000
   manual_check       0.00      0.00      0.00         0
       mercedes       0.99      0.98      0.99    120000
unknown_article       0.98      0.93      0.96    120000
            vag       0.99      0.93      0.96    120000

       accuracy                           0.95    480000
      macro avg       0.79      0.76      0.77    480000
   weighted avg       0.99      0.95      0.97    480000



## TopBFM — Cluster Analysis

In [5]:
with open(_CLUSTER_JSON, encoding="utf-8") as f:
    cluster_data = json.load(f)

rows = []
for cid, info in cluster_data.items():
    size   = info["size"]
    purity = max(info["counts"].values()) / size if size > 0 else 0
    rows.append({"cluster": int(cid), "label": info["label"], "size": size, "purity": round(purity, 4)})
cdf = pd.DataFrame(rows).sort_values("cluster")

# Purity histogram
fig1 = px.histogram(
    cdf, x="purity", color="label", nbins=40,
    title="TopBFM — cluster purity distribution",
    barmode="overlay", opacity=0.7,
)
fig1.add_vline(x=0.92, line_dash="dash", annotation_text="threshold 0.92")
fig1.show()

# Purity vs index (bubble = size)
fig2 = px.scatter(
    cdf, x="cluster", y="purity", size="size", color="label",
    title="TopBFM — purity vs cluster index  (bubble = cluster size)",
    hover_data=["size"],
)
fig2.show()

clean = (cdf["purity"] >= 0.92).sum()
print(f"Total: {len(cdf)}  |  Clean (≥0.92): {clean} ({clean/len(cdf)*100:.1f}%)  |  Avg purity: {cdf['purity'].mean():.3f}")
print("Labels:", dict(Counter(cdf["label"])))

Total: 550  |  Clean (≥0.92): 514 (93.5%)  |  Avg purity: 0.977
Labels: {'bmw': 118, 'unknown_article': 130, 'mercedes': 128, 'vag': 138, 'manual_check': 36}


## TopBFM — 1M File Results

In [6]:
topbfm_df = pd.read_csv(_LABELED_TOPBFM, dtype=str)
total = len(topbfm_df)

counts = topbfm_df["label"].value_counts().reset_index()
counts.columns = ["label", "count"]
counts["pct"] = (counts["count"] / total * 100).round(2)

fig = px.bar(
    counts, x="label", y="count", text="pct",
    title=f"TopBFM — label distribution ({total:,} articles)",
    color="label",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()

prob_cols = [c for c in topbfm_df.columns if c.endswith("_prob")]
if prob_cols:
    conf_rows = []
    for lbl in topbfm_df["label"].unique():
        sub = topbfm_df[topbfm_df["label"] == lbl][prob_cols].apply(pd.to_numeric, errors="coerce")
        mp  = sub.max(axis=1).dropna()
        if len(mp) == 0: continue
        conf_rows.append({"label": lbl, "mean": mp.mean(), "p10": mp.quantile(0.1), "p90": mp.quantile(0.9)})
    cdf = pd.DataFrame(conf_rows)
    cdf["err_high"] = cdf["p90"] - cdf["mean"]
    cdf["err_low"]  = cdf["mean"] - cdf["p10"]
    fig2 = px.bar(
        cdf, x="label", y="mean",
        error_y="err_high", error_y_minus="err_low",
        title="TopBFM — mean confidence per label  [error bars: p10–p90]",
        color="label",
    )
    fig2.show()